# 09 · Webcam and RTSP streams

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microsoft/SPARROW-Playground/blob/main/notebooks/09_webcam.ipynb)

Run detection live on a local camera (`source=0`) or a network stream
(`source="rtsp://..."`). This needs a machine with a camera, so it does not work on
Colab. Set `RUN_WEBCAM = True` to start.

In [ ]:
# Setup: install sparrow-engine if it is missing (for example on Colab).
# For an NVIDIA GPU, install `sparrow-engine-gpu` instead (same Python API).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("sparrow_engine") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sparrow-engine[video]>=0.1.31"])

import sparrow_engine as spe

print("sparrow-engine", spe.__version__, "| device:", spe.active_device())

In [ ]:
# Models are installed into SPARROW_ENGINE_MODEL_DIR (default ~/.sparrow-engine/models).
# Already-installed models are skipped. Inference never downloads anything by itself.
for model_id in ["MDV6-yolov10-c"]:
    spe.download_model(model_id)

In [ ]:
import os

RUN_WEBCAM = os.environ.get("SPP_RUN_WEBCAM") == "1"  # or set to True
SOURCE = 0            # camera index, or an RTSP/HTTP URL
MAX_FRAMES = 100      # None runs until interrupted

In [ ]:
from IPython.display import Image, clear_output, display
import cv2

if not RUN_WEBCAM:
    print("RUN_WEBCAM is False; skipping.")
else:
    try:
        for f in spe.detect_webcam(SOURCE, model="MDV6-yolov10-c", threshold=0.4,
                                   every_n=2, max_frames=MAX_FRAMES):
            frame = spe.draw_detections(f.frame, f.result)
            ok, jpg = cv2.imencode(".jpg", frame)
            clear_output(wait=True)
            display(Image(data=jpg.tobytes(), width=640))
            print(f"t={f.timestamp_s:.1f}s  detections={len(f.result.detections)}")
    except KeyboardInterrupt:
        pass

Add `classifier="..."` to classify each box, as in notebook 08.
Use `scripts/webcam.py` to run the same loop in an OpenCV window outside Jupyter.